# Edifícios no DuckDB (GeoParquet)

Ler pegadas de edifícios direto do S3, agregar por grade S2 e recortar Lesoto
por uma área de interesse. O DuckDB espacial não usa a GPU desta máquina.
O ganho está em uma conexão só, com todos os núcleos, em vez de duas conexões
que não enxergam as tabelas uma da outra.

O arquivo do Brasil tem dezenas de milhões de linhas. A célula que cria
`brasil` é a lenta. Austrália e Lesoto são menores.

In [1]:
import os
import duckdb

con = duckdb.connect()
con.execute(f"SET threads = {os.cpu_count()}")
con.install_extension("httpfs")
con.load_extension("httpfs")
con.install_extension("spatial")
con.load_extension("spatial")
print("threads:", con.execute("SELECT current_setting('threads')").fetchone()[0])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

threads: 20


## Brasil

`parquet_scan` lê os GeoParquet particionados por país sem baixar o bucket inteiro.
`CREATE OR REPLACE` permite rodar a célula de novo.

In [2]:
PREFIXO = "s3://us-west-2.opendata.source.coop/vida/google-microsoft-open-buildings/geoparquet"

con.execute(f"""
    CREATE OR REPLACE TABLE brasil AS
    SELECT * FROM parquet_scan('{PREFIXO}/by_country_s2/country_iso=BRA/*.parquet')
""")
con.execute("SELECT bf_source AS fonte, COUNT(*) AS n FROM brasil GROUP BY 1").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,fonte,n
0,google,136237330
1,microsoft,4807793


## Austrália por grade S2

Cada arquivo é uma célula S2. A média de edifícios por célula resume a densidade
sem trazer a geometria de cada prédio para o notebook.

In [3]:
con.execute(f"""
    CREATE OR REPLACE TABLE australia AS
    SELECT s2_id, COUNT(geometry) AS n
    FROM parquet_scan('{PREFIXO}/by_country_s2/country_iso=AUS/*.parquet')
    GROUP BY s2_id
""")
con.execute("SELECT ROUND(AVG(n), 0) AS media_por_grade FROM australia").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,media_por_grade
0,4027416.0


## Lesoto: fontes e recorte

Pergunta de negócio: na área de `limites.geojson`, o inventário mesclado
(Google + Microsoft) traz quantos edifícios a mais do que o Open Buildings
só do Google?

Lesoto cabe na memória. Recortamos as duas fontes pela AOI (`ST_Intersects` /
`ST_Intersection`) e contamos o que aparece no mesclado e não no Google sozinho.

In [13]:
con.execute(f"""
    CREATE OR REPLACE TABLE lesoto AS
    SELECT * FROM parquet_scan('{PREFIXO}/by_country_s2/country_iso=LSO/*.parquet')
""")
print("Lesoto inteiro — mesclado (Google + Microsoft) por fonte:")
print(con.execute("SELECT bf_source, COUNT(*) AS n FROM lesoto GROUP BY 1").df())

google = "s3://us-west-2.opendata.source.coop/google-research-open-buildings/geoparquet-by-country"
con.execute(f"""
    CREATE OR REPLACE TABLE lesoto_google AS
    SELECT * FROM '{google}/country_iso=LS/LS.parquet'
""")
print(
    "Open Buildings só Google (país):",
    con.execute("SELECT COUNT(*) FROM lesoto_google").fetchone()[0],
)
print(
    "Parte Google dentro do mesclado (país):",
    con.execute("SELECT COUNT(*) FROM lesoto WHERE bf_source = 'google'").fetchone()[0],
)

con.execute("CREATE OR REPLACE TABLE aoi AS SELECT * FROM ST_Read('limites.geojson')")
con.execute("""
    CREATE OR REPLACE TABLE lesoto_recorte AS
    SELECT ST_Intersection(b.geometry, a.geom) AS geom, b.bf_source
    FROM lesoto b, aoi a
    WHERE ST_Intersects(b.geometry, a.geom)
""")
con.execute("""
    CREATE OR REPLACE TABLE lesoto_google_recorte AS
    SELECT ST_Intersection(b.geometry, a.geom) AS geom
    FROM lesoto_google b, aoi a
    WHERE ST_Intersects(b.geometry, a.geom)
""")
con.execute("""
    CREATE OR REPLACE TABLE so_no_mesclado AS
    SELECT m.*
    FROM lesoto_recorte m
    WHERE NOT EXISTS (
        SELECT 1 FROM lesoto_google_recorte g
        WHERE ST_Intersects(m.geom, g.geom)
    )
""")
n_mesclado = con.execute("SELECT COUNT(*) FROM lesoto_recorte").fetchone()[0]
n_google = con.execute("SELECT COUNT(*) FROM lesoto_google_recorte").fetchone()[0]
n_extra = con.execute("SELECT COUNT(*) FROM so_no_mesclado").fetchone()[0]
print("--- Na área de interesse (limites.geojson) ---")
print("Edifícios no inventário mesclado:", n_mesclado)
print("Edifícios só no Open Buildings Google:", n_google)
print(
    "Cobertura extra do mesclado (não aparece no Google sozinho):",
    n_extra,
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Lesoto inteiro — mesclado (Google + Microsoft) por fonte:
   bf_source        n
0     google  1394190
1  microsoft   151722


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Open Buildings só Google (país): 1394225
Parte Google dentro do mesclado (país): 1394190
--- Na área de interesse (limites.geojson) ---
Edifícios no inventário mesclado: 13561
Edifícios só no Open Buildings Google: 13213
Cobertura extra do mesclado (não aparece no Google sozinho): 348


## Exportar

FlatGeobuf guarda geometria pronta para QGIS. O segundo arquivo leva atributos
do Lesoto inteiro, não só o recorte.

In [5]:
con.execute("""
    COPY (SELECT * FROM lesoto_recorte)
    TO 'dataset1.fgb' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")
con.execute("""
    COPY (
        SELECT boundary_id, confidence, area_in_meters, geometry
        FROM lesoto
    ) TO 'dataset2.fgb' WITH (FORMAT GDAL, DRIVER 'FlatGeobuf')
""")
print("dataset1.fgb e dataset2.fgb gravados")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dataset1.fgb e dataset2.fgb gravados


## Mapa da área de interesse

`limites.geojson` sobre tiles do Google Maps; abre em `http://127.0.0.1`.

In [12]:
import threading
import webbrowser
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import folium

# Google tiles: m=rua, s=satélite, y=híbrido. Sem API key (pode falhar se o Google bloquear).
TILES = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
mapa = folium.Map(tiles=TILES, attr="Google")
area = folium.GeoJson("limites.geojson", name="limites").add_to(mapa)
mapa.fit_bounds(area.get_bounds())

out = Path("mapa_limites.html").resolve()
mapa.save(out)

url = f"http://127.0.0.1:8765/{out.name}"
try:
    handler = partial(SimpleHTTPRequestHandler, directory=str(out.parent))
    httpd = ThreadingHTTPServer(("127.0.0.1", 8765), handler)
    threading.Thread(target=httpd.serve_forever, daemon=True).start()
except OSError:
    pass  # porta já em uso de uma execução anterior
webbrowser.open(url)
print("mapa aberto:", url)

mapa aberto: http://127.0.0.1:8765/mapa_limites.html
